# Procesamiento de Video con YOLO

Este notebook está diseñado únicamente para procesar un video utilizando los 3 modelos integrados:
1. **Detección OBB** (modelo personalizado de tenis de mesa entrenado con YOLO26 OBB).
2. **Segmentación de Objetos** (yolo26n-seg.pt de COCO, excluyendo personas).
3. **Pose Estimation** (yolo26n-pose.pt para el esqueleto de los jugadores).

El procesamiento está limitado para extraer y analizar únicamente **10 segundos** del video original.

### 1. Instalación de Librerías
Instalamos los paquetes necesarios para procesar video e inferencia de modelos.

In [1]:
%pip install ultralytics opencv-python numpy pyyaml matplotlib

Note: you may need to restart the kernel to use updated packages.


### 2. Parámetros de Confianza


In [2]:
# Duración máxima a procesar
SEGUNDOS_A_PROCESAR = 20

# Umbrales de confianza mínimos para cada modelo 
CONF_DETECTION_CLASS = {
    'TT TABLE': 0.80,   # Mesa
    'TT NET': 0.50,     # Red
    'TT RACKET': 0.35   # Paleta
}
MIN_CONF_DETECTION = min(CONF_DETECTION_CLASS.values())
CONF_SEG = 0.30  # Para el modelo de segmentación de objetos COCO
CONF_POSE = 0.25  # Para el modelo de pose 

print("Parámetros de confianza configurados:")
print(f"Detección OBB (Mínima): {MIN_CONF_DETECTION}")
print(f"Segmentación: {CONF_SEG}")
print(f"Pose: {CONF_POSE}")

Parámetros de confianza configurados:
Detección OBB (Mínima): 0.35
Segmentación: 0.3
Pose: 0.25


### 3. Cargar Modelos
Buscamos el mejor modelo entrenado (`best.pt`) y cargamos los modelos preentrenados de Segmentación y Pose de **YOLO26**.

In [ ]:
import os
import glob
from collections import defaultdict
# import numpy as np
import cv2
from ultralytics import YOLO

# Buscar dinámicamente los mejores pesos entrenados
best_weights = '../runs/obb/tenis_mesa_obb_run/weights/best.pt'
if best_weights:
    BEST_WEIGHTS_PATH = best_weights
    print(f"Modelo personalizado cargado desde: {BEST_WEIGHTS_PATH}")
else:
    BEST_WEIGHTS_PATH = 'yolo26n-obb.pt'
    print(f"No se encontró best.pt, usando modelo base: {BEST_WEIGHTS_PATH}")

model_detection    = YOLO(BEST_WEIGHTS_PATH)
model_segmentation = YOLO('yolo26n-seg.pt')
model_pose         = YOLO('yolo26n-pose.pt')

print("Modelos cargados correctamente")

Modelo personalizado cargado desde: ../runs/obb/tenis_mesa_obb_run/weights/best.pt
Modelos cargados correctamente


### 4. Configuración del Video de Entrada/Salida
Busca los videos disponibles en el directorio `Videos/` y configura el video resultante de 10 segundos.

In [31]:
# Crear carpeta para videos procesados si no existe
os.makedirs('../data/videos/processed', exist_ok=True)

# Nombre del video a buscar
VIDEO_NAME = 'demo_01.mp4'

# Buscar videos en la carpeta data/videos/
videos = glob.glob(f'../data/videos/original/{VIDEO_NAME}')

if videos:
    VIDEO_INPUT = videos[0]
    print(f"Video encontrado para procesar: {VIDEO_INPUT}")
else:
    # Fallback: buscar cualquier mp4
    all_videos = glob.glob('data/videos/*.mp4')
    if all_videos:
        VIDEO_INPUT = all_videos[0]
        print(f"No se encontró '{VIDEO_NAME}', procesando el primer video encontrado: {VIDEO_INPUT}")
    else:
        VIDEO_INPUT = VIDEO_NAME
        print(f"No se encontró el video '{VIDEO_NAME}' ni otros en data/videos/, usando fallback: {VIDEO_INPUT}")

# Guardar el video procesado dentro de la carpeta 'data/videos/processed'
VIDEO_OUTPUT = os.path.join('../data/videos/processed/', f'processed_{os.path.basename(VIDEO_INPUT)}')
print(f"El video procesado se guardará en: {VIDEO_OUTPUT}")

Video encontrado para procesar: ../data/videos/original/demo_01.mp4
El video procesado se guardará en: ../data/videos/processed/processed_demo_01.mp4


### 5. Definición de Funciones de Dibujo (Visualización)
Configuramos colores personalizados y lógica para dibujar bounding boxes orientadas (OBB), segmentación COCO (sin personas), pose estimation e información en pantalla.

In [32]:
# Configuración de Clases y Panel HUD
import time

# ID de la clase de persona en COCO (para excluirla de la segmentación)
COCO_PERSON_CLASS_ID = 0

# Lista de clases para la segmentación (todas excepto personas)
CLASES_SEG_INCLUIDAS = [i for i in range(80) if i != COCO_PERSON_CLASS_ID]

# Colores para nuestras clases personalizadas
CUSTOM_CLASS_COLORS = {
    'tt net':    (252, 191, 73),   # Naranja/Amarillo
    'tt racket': (0, 180, 216),    # Celeste
    'tt table':  (255, 0, 110),    # Rosa
}

# Nombres a mostrar en el panel
DISPLAY_NAMES = {
    'tt net':    'TT Net',
    'tt racket': 'TT Racket',
    'tt table':  'TT Table',
}
ALL_CUSTOM_CLASSES = ['TT Racket', 'TT Net', 'TT Table']


def extraer_conteo_detecciones(det_result, conf_thresh_dict):
    # Cuenta las detecciones filtrando por confianza
    counts = defaultdict(int)

    is_obb = hasattr(det_result, 'obb') and det_result.obb is not None and len(det_result.obb) > 0
    is_boxes = hasattr(det_result, 'boxes') and det_result.boxes is not None and len(det_result.boxes) > 0

    if not is_obb and not is_boxes:
        return dict(counts)

    detections = det_result.obb if is_obb else det_result.boxes

    for box in detections:
        cls_id     = int(box.cls[0])
        confidence = float(box.conf[0])
        cls_name   = det_result.names[cls_id]

        # Umbral especifico por clase
        normalized_name = cls_name.upper().strip()
        thresh = conf_thresh_dict.get(normalized_name, 0.25)
        if confidence < thresh:
            continue

        normalized = cls_name.lower().strip()
        display    = DISPLAY_NAMES.get(normalized, cls_name)
        counts[display] += 1

    return dict(counts)


def draw_info_panel(frame, counts, fps, frame_idx, n_persons):
    # Dibuja un HUD semitransparente con los conteos y FPS
    h, w = frame.shape[:2]

    # Tamaño y padding del panel
    line_h     = 30
    pad_x      = 16
    pad_y      = 14
    header_h   = 36
    sep_h      = 12
    dot_radius = 5

    n_lines = len(ALL_CUSTOM_CLASSES) + 1
    panel_h = pad_y + header_h + sep_h + (n_lines * line_h) + sep_h + line_h + pad_y
    panel_w = 260

    x0, y0 = 12, 12

    # Fondo semitransparente
    overlay = frame.copy()
    cv2.rectangle(overlay, (x0, y0), (x0 + panel_w, y0 + panel_h), (18, 18, 24), -1)
    cv2.rectangle(overlay, (x0, y0), (x0 + panel_w, y0 + panel_h), (55, 55, 65), 1)
    cv2.addWeighted(overlay, 0.78, frame, 0.22, 0, frame)

    # Título del panel
    header_y = y0 + pad_y + 20
    cv2.putText(
        frame, "DETECCIONES EN TIEMPO REAL",
        (x0 + pad_x, header_y),
        cv2.FONT_HERSHEY_SIMPLEX, 0.48, (200, 200, 210), 1, cv2.LINE_AA
    )

    # Separador
    sep_y = header_y + sep_h
    cv2.line(
        frame,
        (x0 + pad_x, sep_y), (x0 + panel_w - pad_x, sep_y),
        (70, 70, 80), 1
    )

    # Mostrar clases de tenis de mesa
    y = sep_y + 24
    for cls_name in ALL_CUSTOM_CLASSES:
        count = counts.get(cls_name, 0)
        color = CUSTOM_CLASS_COLORS.get(cls_name.lower().strip(), (200, 200, 200))

        cv2.circle(frame, (x0 + pad_x + dot_radius, y - 4), dot_radius, color, -1, cv2.LINE_AA)
        text = f"{cls_name}: {count}"
        cv2.putText(
            frame, text,
            (x0 + pad_x + dot_radius * 2 + 10, y),
            cv2.FONT_HERSHEY_SIMPLEX, 0.50, (235, 235, 240), 1, cv2.LINE_AA
        )
        y += line_h

    # Conteo de personas
    cv2.circle(frame, (x0 + pad_x + dot_radius, y - 4), dot_radius, (0, 255, 255), -1, cv2.LINE_AA)
    cv2.putText(
        frame, f"Personas: {n_persons}",
        (x0 + pad_x + dot_radius * 2 + 10, y),
        cv2.FONT_HERSHEY_SIMPLEX, 0.50, (0, 255, 255), 1, cv2.LINE_AA
    )
    y += line_h

    # FPS y número de frame
    sep_y2 = y + 2
    cv2.line(
        frame,
        (x0 + pad_x, sep_y2), (x0 + panel_w - pad_x, sep_y2),
        (50, 50, 60), 1
    )
    y = sep_y2 + 20
    cv2.putText(
        frame, f"FPS: {fps:.1f}  |  Frame: {frame_idx}",
        (x0 + pad_x, y),
        cv2.FONT_HERSHEY_SIMPLEX, 0.40, (120, 120, 135), 1, cv2.LINE_AA
    )

    return frame


print("Funciones de visualización y HUD cargadas correctamente")


Funciones de visualización y HUD cargadas correctamente


### 6. Pipeline de Procesamiento (Límite 10 segundos)
Corremos el pipeline que lee el video, ejecuta inferencia con los 3 modelos, ensambla el panel de visualización y guarda los primeros 10 segundos en un nuevo archivo.

In [33]:
import json

cap = cv2.VideoCapture(VIDEO_INPUT)
if not cap.isOpened():
    raise FileNotFoundError(f"No se pudo abrir el video: {VIDEO_INPUT}")

fps_in  = cap.get(cv2.CAP_PROP_FPS) or 30
width   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))


limit_frames = min(int(SEGUNDOS_A_PROCESAR * fps_in), total)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out    = cv2.VideoWriter(VIDEO_OUTPUT, fourcc, fps_in, (width, height))

frame_idx = 0
json_frames = []
total_inference_time = 0
custom_conf_sum = 0
custom_conf_count = 0
print(f"Procesando hasta {SEGUNDOS_A_PROCESAR}s de video ({limit_frames} frames)...\n")

try:
    while cap.isOpened() and frame_idx < limit_frames:
        ret, frame = cap.read()
        if not ret:
            break

        t0 = time.time()

        # Paso 1: Segmentación (excluyendo personas)
        result_seg = model_segmentation.predict(
            frame,
            conf=CONF_SEG,
            iou=0.45,
            imgsz=640,
            classes=CLASES_SEG_INCLUIDAS,
            verbose=False
        )[0]

        # Graficar segmentación sin bboxes para no sobrecargar
        frame_seg = result_seg.plot(
            boxes=False,
            labels=True,
            conf=True
        )

        # Paso 2: Pose estimation (solo para los jugadores)
        result_pose = model_pose.predict(
            frame_seg,
            conf=CONF_POSE,
            iou=0.45,
            imgsz=640,
            verbose=False
        )[0]

        # Contamos personas
        n_persons = 0
        if result_pose.boxes is not None:
            n_persons = sum(1 for c in result_pose.boxes.conf if float(c) >= CONF_POSE)

        # Dibujar keypoints y esqueleto
        frame_pose = result_pose.plot(
            boxes=False,
            labels=False,
            conf=False,
            kpt_radius=5
        )

        # Paso 3: Detección de elementos con modelo personalizado (OBB)
        result_det = model_detection.predict(
            frame_pose,
            conf=MIN_CONF_DETECTION,
            iou=0.25,
            imgsz=640,
            verbose=False
        )[0]

        # Filtrar detecciones según su confianza específica por clase para que no se cuenten ni se dibujen
        if len(result_det) > 0:
            keep = []
            detections = result_det.obb if (hasattr(result_det, 'obb') and result_det.obb is not None and len(result_det.obb) > 0) else result_det.boxes
            if detections is not None:
                for i, box in enumerate(detections):
                    cls_id = int(box.cls[0])
                    confidence = float(box.conf[0])
                    cls_name = result_det.names[cls_id].upper().strip()
                    thresh = CONF_DETECTION_CLASS.get(cls_name, 0.25)
                    if confidence >= thresh:
                        keep.append(i)
                result_det = result_det[keep]

        custom_counts = extraer_conteo_detecciones(result_det, CONF_DETECTION_CLASS)

        # --- Extracción de datos para JSON ---
        frame_data = {'detections': [], 'poses': [], 'segmentations': []}
        
        if hasattr(result_seg, 'masks') and result_seg.masks is not None:
            for i, mask in enumerate(result_seg.masks.xy):
                cls_id = int(result_seg.boxes.cls[i])
                conf = float(result_seg.boxes.conf[i])
                cls_name = result_seg.names[cls_id]
                frame_data['segmentations'].append({
                    'polygon': mask.tolist(),
                    'class_name': cls_name,
                    'confidence': conf
                })
                
        if hasattr(result_pose, 'keypoints') and result_pose.keypoints is not None:
            kpts_data = result_pose.keypoints.data
            for i in range(len(kpts_data)):
                person_kpts = []
                for kpt in kpts_data[i]:
                    person_kpts.append({'x': float(kpt[0]), 'y': float(kpt[1]), 'confidence': float(kpt[2])})
                frame_data['poses'].append({'keypoints': person_kpts})
                
        final_detections = result_det.obb if (hasattr(result_det, 'obb') and result_det.obb is not None and len(result_det.obb) > 0) else result_det.boxes
        if final_detections is not None:
            for i, box in enumerate(final_detections):
                cls_id = int(box.cls[0])
                conf = float(box.conf[0])
                cls_name = result_det.names[cls_id]
                if hasattr(box, 'xyxyxyxy'):
                    pts = box.xyxyxyxy[0].cpu().numpy()
                    x1, y1 = float(pts[:, 0].min()), float(pts[:, 1].min())
                    x2, y2 = float(pts[:, 0].max()), float(pts[:, 1].max())
                else:
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().tolist()
                frame_data['detections'].append({'bbox': [x1, y1, x2, y2], 'class_name': cls_name, 'confidence': conf})
                custom_conf_sum += conf
                custom_conf_count += 1
                
        json_frames.append(frame_data)
        inference_time_ms = (time.time() - t0) * 1000
        total_inference_time += inference_time_ms
        # -------------------------------------

        # Graficar bboxes del modelo OBB
        frame_det = result_det.plot(
            boxes=True,
            labels=True,
            conf=True,
            line_width=2
        )

        # Paso 4: Agregar HUD informativo
        elapsed_fps = 1.0 / (time.time() - t0 + 1e-9)
        frame_final = draw_info_panel(
            frame_det,
            custom_counts,
            elapsed_fps,
            frame_idx,
            n_persons
        )

        # Guardar frame
        out.write(frame_final)
        frame_idx += 1

        # Reporte de progreso cada 30 frames
        if frame_idx % 30 == 0:
            pct = frame_idx / limit_frames * 100
            print(f"  Frame {frame_idx:04d}/{limit_frames} ({pct:.0f}%) — {elapsed_fps:.1f} FPS")

finally:
    cap.release()
    out.release()

video_info = {
    'fps': fps_in,
    'width': width,
    'height': height,
    'custom_classes_conf_avg': (custom_conf_sum / custom_conf_count) if custom_conf_count > 0 else 0,
    'inference_time_ms_avg': (total_inference_time / frame_idx) if frame_idx > 0 else 0,
    'total_frames': frame_idx
}
json_output_data = {'video_info': video_info, 'frames': json_frames}
json_path = VIDEO_OUTPUT.replace('.mp4', '_data.json')
os.makedirs(os.path.dirname(json_path), exist_ok=True)
with open(json_path, 'w') as f:
    json.dump(json_output_data, f)

print(f"\nCompletado. Video procesado guardado en: {VIDEO_OUTPUT}")
print(f"JSON de datos guardado en: {json_path}")
print(f"Total de frames procesados: {frame_idx}")


Procesando hasta 20s de video (1038 frames)...

  Frame 0030/1038 (3%) — 2.8 FPS
  Frame 0060/1038 (6%) — 3.1 FPS
  Frame 0090/1038 (9%) — 2.6 FPS
  Frame 0120/1038 (12%) — 3.0 FPS
  Frame 0150/1038 (14%) — 3.0 FPS
  Frame 0180/1038 (17%) — 3.2 FPS
  Frame 0210/1038 (20%) — 3.0 FPS
  Frame 0240/1038 (23%) — 3.2 FPS
  Frame 0270/1038 (26%) — 3.3 FPS
  Frame 0300/1038 (29%) — 3.5 FPS
  Frame 0330/1038 (32%) — 3.8 FPS
  Frame 0360/1038 (35%) — 3.1 FPS
  Frame 0390/1038 (38%) — 2.8 FPS
  Frame 0420/1038 (40%) — 2.3 FPS
  Frame 0450/1038 (43%) — 2.7 FPS
  Frame 0480/1038 (46%) — 3.6 FPS
  Frame 0510/1038 (49%) — 2.9 FPS
  Frame 0540/1038 (52%) — 3.2 FPS
  Frame 0570/1038 (55%) — 3.5 FPS
  Frame 0600/1038 (58%) — 3.5 FPS
  Frame 0630/1038 (61%) — 3.2 FPS
  Frame 0660/1038 (64%) — 3.4 FPS
  Frame 0690/1038 (66%) — 3.1 FPS
  Frame 0720/1038 (69%) — 3.0 FPS
  Frame 0750/1038 (72%) — 3.1 FPS
  Frame 0780/1038 (75%) — 3.0 FPS
  Frame 0810/1038 (78%) — 3.2 FPS
  Frame 0840/1038 (81%) — 3.3 FPS
  F